In [1]:
#使用するライブラリをimport
from pprint import pprint

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score,mean_absolute_error
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import train_test_split
import lightgbm as lgb

#データセットの読み込み
path = '../data/raw/Milling_Tool_Dataset.csv'
df = pd.read_csv(path)

#訓練用、テスト用、検証用にデータを分割
X = df.drop(["cutting_speed", "feed_rate", "material_hardness", "tool_wear", "RUL"], axis=1)
y = df["tool_wear"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

#2つのモデルを比較
models = {
    "RandomForest": RandomForestRegressor(n_estimators=200, random_state=42),
    "LightGBM": lgb.LGBMRegressor(n_estimators=500, learning_rate=0.05, random_state=42)
}

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, scoring="neg_mean_absolute_error", cv=5)
    print(f"{name}: {scores.mean():.4f} (+/- {scores.std():.4f})")

RandomForest: -1.9921 (+/- 0.1450)
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000157 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 784, number of used features: 5
[LightGBM] [Info] Start training from score 6.977890
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No fu

RandamForestとLightGBMのモデルを比較。

RandamForestのほうが評価が良いが、LightGBMのほうがばらつきが小さい。

両モデルにハイパーパラメータ調整を行う。

In [2]:
#RandomForestのハイパーパラメータ調整
from sklearn.model_selection import GridSearchCV

search_params = {
    'n_estimators'      : [20, 50, 100, 300],
    'random_state'      : [42],
    'n_jobs'            : [-1],
    'min_samples_split' : [5, 10, 25, 50, 100],
    'max_depth'         : [5, 10, 25, 50, 100]
}

cv = GridSearchCV(RandomForestRegressor(),search_params,verbose=2)

cv.fit(X_train, y_train)

Fitting 5 folds for each of 100 candidates, totalling 500 fits
[CV] END max_depth=5, min_samples_split=5, n_estimators=20, n_jobs=-1, random_state=42; total time=   0.0s
[CV] END max_depth=5, min_samples_split=5, n_estimators=20, n_jobs=-1, random_state=42; total time=   0.0s
[CV] END max_depth=5, min_samples_split=5, n_estimators=20, n_jobs=-1, random_state=42; total time=   0.0s
[CV] END max_depth=5, min_samples_split=5, n_estimators=20, n_jobs=-1, random_state=42; total time=   0.0s
[CV] END max_depth=5, min_samples_split=5, n_estimators=20, n_jobs=-1, random_state=42; total time=   0.0s
[CV] END max_depth=5, min_samples_split=5, n_estimators=50, n_jobs=-1, random_state=42; total time=   0.0s
[CV] END max_depth=5, min_samples_split=5, n_estimators=50, n_jobs=-1, random_state=42; total time=   0.0s
[CV] END max_depth=5, min_samples_split=5, n_estimators=50, n_jobs=-1, random_state=42; total time=   0.0s
[CV] END max_depth=5, min_samples_split=5, n_estimators=50, n_jobs=-1, random_sta

,estimator,RandomForestRegressor()
,param_grid,"{'max_depth': [5, 10, ...], 'min_samples_split': [5, 10, ...], 'n_estimators': [20, 50, ...], 'n_jobs': [-1], ...}"
,scoring,None
,n_jobs,None
,refit,True
,cv,None
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,300


In [3]:
#調整済みモデルで予測
RFR_model=cv.best_estimator_
train_predict = RFR_model.predict(X_train)
test_predict = RFR_model.predict(X_test)

#評価
result_list=[]
MAE_train  =mean_absolute_error(y_train, train_predict)
MAE_test  =mean_absolute_error(y_test, test_predict)
R2_train = r2_score(y_train, train_predict)
R2_test = r2_score(y_test, test_predict)
result = {
    "model":"RandomForest",
    "dataset":"raw_data",
    "MAE_train":mean_absolute_error(y_train, train_predict),
    "MAE_test":mean_absolute_error(y_test, test_predict),
    "R^2_train":r2_score(y_train, train_predict),
    "R^2_test":r2_score(y_test, test_predict)
}

pprint(result)
result_list.append(result)

{'MAE_test': 1.8606068638591144,
 'MAE_train': 1.6736937495213469,
 'R^2_test': 0.638382047694643,
 'R^2_train': 0.7384181549040346,
 'dataset': 'raw_data',
 'model': 'RandomForest'}


In [4]:
#LightGBMのハイパーパラメータ調整
params = {
    'objective': 'regression',
    'metric': "mae",
    'num_leaves': 31,         # default = 31,
    'learning_rate': 0.1,    # default = 0.1
    'feature_fraction': 1.0,  # default = 1.0
    'bagging_freq': 0,        # default = 0
    'bagging_fraction': 1.0,  # default = 1.0
    'random_state': 0,        # default = None
}
num_round = 100

trains = lgb.Dataset(X_train, y_train)
tests = lgb.Dataset(X_test, y_test)

import optuna
import optuna.integration.lightgbm as lgb_tuner

model = lgb_tuner.LightGBMTuner(
    params, trains,valid_sets=[tests],
    num_boost_round=num_round,)

model.run()

best_params = model.best_params
print("Best params:", best_params)

LGBM_model = model.get_best_booster()

[I 2025-11-05 11:33:56,372] A new study created in memory with name: no-name-6320399e-9493-4fb5-bf83-bff190d26ccc
feature_fraction, val_score: inf:   0%|                   | 0/7 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000134 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 1.924412:  14%|8     | 1/7 [00:00<00:01,  3.28it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000133 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 1.924412:  29%|#7    | 2/7 [00:00<00:01,  3.25it/s]

...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000092 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572


min_child_samples, val_score: 1.827849:  80%|#### | 4/5 [00:00<00:00, 32.88it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000098 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000070 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000144 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training fro

min_child_samples, val_score: 1.827849: 100%|#####| 5/5 [00:00<00:00, 32.74it/s]

Best params: {'objective': 'regression', 'metric': 'l1', 'num_leaves': 2, 'learning_rate': 0.1, 'feature_fraction': 1.0, 'bagging_freq': 4, 'bagging_fraction': 0.4002383346399543, 'random_state': 0, 'feature_pre_filter': False, 'lambda_l1': 0.10714036029509091, 'lambda_l2': 0.0004118418641094273, 'min_child_samples': 5}


In [5]:
train_predict = LGBM_model.predict(X_train)
test_predict = LGBM_model.predict(X_test)
MAE_train  =mean_absolute_error(y_train, train_predict)
MAE_test  =mean_absolute_error(y_test, test_predict)
R2_train = r2_score(y_train, train_predict)
R2_test = r2_score(y_test, test_predict)
result = {
    "model":"LightGBM",
    "dataset":"raw_data",
    "MAE_train":mean_absolute_error(y_train, train_predict),
    "MAE_test":mean_absolute_error(y_test, test_predict),
    "R^2_train":r2_score(y_train, train_predict),
    "R^2_test":r2_score(y_test, test_predict)
}

pprint(result)
result_list.append(result)

{'MAE_test': 1.8278489561335856,
 'MAE_train': 1.8059772294999925,
 'R^2_test': 0.6335932030198868,
 'R^2_train': 0.6872437169534038,
 'dataset': 'raw_data',
 'model': 'LightGBM'}


パラメータ調整済みモデルは、どちらもR^2(決定係数)=0.63〜0.64, MAE(平均絶対誤差)=1.8〜1.9とあまり変わらない。

計算負荷が低く、ばらつきの少なかったLightGBMを使用して以後の分析を続行する。

In [6]:
#各特徴量のモデル作成時の寄与とモデルの予測精度への寄与を評価
from sklearn.inspection import permutation_importance
regressor_model = lgb.LGBMRegressor()
regressor_model._Booster = LGBM_model
regressor_model._n_features = X_test.shape[1]
regressor_model.fitted_ = True
results = permutation_importance(regressor_model, X_test, y_test, n_repeats=10, random_state=42)

importances_df=pd.DataFrame(zip(X.columns, LGBM_model.feature_importance(importance_type='gain'), results['importances'].mean(axis=1)), columns=['features', 'feature_importance', 'permutation_importance'])

#グラフ化
plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
sns.barplot(importances_df, x='features', y='feature_importance', errorbar=None)
plt.xticks(rotation=45)
plt.title('Feature_importance')
plt.tight_layout()

plt.subplot(1,2,2)
sns.barplot(importances_df, x='features', y='permutation_importance', errorbar=None)
plt.xticks(rotation=45)
plt.title('Permutation_importance')
plt.tight_layout()

plt.savefig("../outputs/figures/modeling/lightgbm_feature_importances_histplot.png", format="png")
plt.close()

importances_df

,features,feature_importance,permutation_importance
0,vibration_x,990.211302,0.020953
1,vibration_y,664.718195,0.003242
2,vibration_z,313.373302,0.004168
3,acoustic_rms,20983.141209,0.888064
4,spindle_load,3685.351107,0.068300


特徴量の重要度評価では、2つの指標はどちらも同じ傾向。"acoustic_rms"の寄与度が非常に大きく評価されていて、続いて"spindle_load", "vibration_x"となっている。これは、eda.ipynbで確認した各センサデータの"tool_wear"との相関係数の値とも矛盾しない。

次に、複数の加工データから移動平均を取ったデータセットでトライする。

In [7]:
#20行のデータの移動平均を使用したデータでトライ
path_avg20 = '../data/processed/mv_avg_20.csv'
df = pd.read_csv(path_avg20)

X = df.drop(["tool_wear"], axis=1)
y = df["tool_wear"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

trains = lgb.Dataset(X_train, y_train)
tests = lgb.Dataset(X_test, y_test)

model = lgb_tuner.LightGBMTuner(
    params, trains,valid_sets=[tests],
    num_boost_round=num_round,)

model.run()

best_params = model.best_params
print("Best params:", best_params)

avg20_model = model.get_best_booster()

[I 2025-11-05 11:35:11,439] A new study created in memory with name: no-name-642cd675-8959-4499-bf5d-57729f3caf10
feature_fraction, val_score: inf:   0%|                   | 0/7 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000175 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 0.271400:  14%|8     | 1/7 [00:00<00:01,  3.24it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000096 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 0.271400:  29%|#7    | 2/7 [00:00<00:01,  3.28it/s]

...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000070 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572


min_child_samples, val_score: 0.241146:  80%|#### | 4/5 [00:01<00:00,  3.10it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000121 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

min_child_samples, val_score: 0.241146: 100%|#####| 5/5 [00:01<00:00,  3.40it/s]

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

In [8]:
train_predict = avg20_model.predict(X_train)
test_predict = avg20_model.predict(X_test)
MAE_train  =mean_absolute_error(y_train, train_predict)
MAE_test  =mean_absolute_error(y_test, test_predict)
R2_train = r2_score(y_train, train_predict)
R2_test = r2_score(y_test, test_predict)
result = {
    "model":"LightGBM",
    "dataset":"mv_avg_20",
    "MAE_train":mean_absolute_error(y_train, train_predict),
    "MAE_test":mean_absolute_error(y_test, test_predict),
    "R^2_train":r2_score(y_train, train_predict),
    "R^2_test":r2_score(y_test, test_predict)
}

pprint(result)
result_list.append(result)

{'MAE_test': 0.24114565130450608,
 'MAE_train': 0.04452207874947638,
 'R^2_test': 0.9908855014307625,
 'R^2_train': 0.9997534665144866,
 'dataset': 'mv_avg_20',
 'model': 'LightGBM'}


In [9]:
#10行のデータの移動平均を使用したデータでトライ
path_avg10 = '../data/processed/mv_avg_10.csv'
df = pd.read_csv(path_avg10)

X = df.drop(["tool_wear"], axis=1)
y = df["tool_wear"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5, random_state=42)

trains = lgb.Dataset(X_train, y_train)
tests = lgb.Dataset(X_test, y_test)

model = lgb_tuner.LightGBMTuner(
    params, trains,valid_sets=[tests],
    num_boost_round=num_round,)

model.run()

best_params = model.best_params
print("Best params:", best_params)

avg10_model = model.get_best_booster()

[I 2025-11-05 11:35:36,560] A new study created in memory with name: no-name-a36fefda-7c3f-4ee1-81a3-c3f12c95a0d9
feature_fraction, val_score: inf:   0%|                   | 0/7 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000181 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 0.576977:  14%|8     | 1/7 [00:00<00:01,  3.18it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000095 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 0.576977:  29%|#7    | 2/7 [00:00<00:01,  3.23it/s]

...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000085 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572


min_child_samples, val_score: 0.564681:  80%|#### | 4/5 [00:00<00:00,  4.10it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000074 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

min_child_samples, val_score: 0.564681: 100%|#####| 5/5 [00:01<00:00,  4.16it/s]

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

In [10]:
train_predict = avg10_model.predict(X_train)
test_predict = avg10_model.predict(X_test)
MAE_train  =mean_absolute_error(y_train, train_predict)
MAE_test  =mean_absolute_error(y_test, test_predict)
R2_train = r2_score(y_train, train_predict)
R2_test = r2_score(y_test, test_predict)
result = {
    "model":"LightGBM",
    "dataset":"mv_avg_10",
    "MAE_train":mean_absolute_error(y_train, train_predict),
    "MAE_test":mean_absolute_error(y_test, test_predict),
    "R^2_train":r2_score(y_train, train_predict),
    "R^2_test":r2_score(y_test, test_predict)
}

pprint(result)
result_list.append(result)

{'MAE_test': 0.5646805983927463,
 'MAE_train': 0.1536216348486473,
 'R^2_test': 0.9612333067196711,
 'R^2_train': 0.9975457868368861,
 'dataset': 'mv_avg_10',
 'model': 'LightGBM'}


In [11]:
#5行のデータの移動平均を使用したデータでトライ
path_avg5 = '../data/processed/mv_avg_5.csv'
df = pd.read_csv(path_avg5)

X = df.drop(["tool_wear"], axis=1)
y = df["tool_wear"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5, random_state=42)

trains = lgb.Dataset(X_train, y_train)
tests = lgb.Dataset(X_test, y_test)

model = lgb_tuner.LightGBMTuner(
    params, trains,valid_sets=[tests],
    num_boost_round=num_round,)

model.run()

best_params = model.best_params
print("Best params:", best_params)

avg5_model = model.get_best_booster()

[I 2025-11-05 11:35:59,896] A new study created in memory with name: no-name-c6952cfa-aa09-4148-8f9f-71a8c45c1237
feature_fraction, val_score: inf:   0%|                   | 0/7 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000165 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 0.922167:  14%|8     | 1/7 [00:00<00:01,  3.16it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000103 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 0.922167:  29%|#7    | 2/7 [00:00<00:01,  3.21it/s]

...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000079 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000109 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] N

min_child_samples, val_score: 0.892764:  60%|###  | 3/5 [00:00<00:00,  9.36it/s]

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000107 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000087 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572


min_child_samples, val_score: 0.892764: 100%|#####| 5/5 [00:00<00:00,  8.86it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000120 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572
Best params: {'objective': 'regression', 'metric': 'l1', 'num_leaves': 10, 'learning_rate': 0.1, 'feature_fraction': 0.7, 'bagging_freq': 1, 'bagging_fraction': 0.9274019754870484, 'random_state': 0, 'feature_pre_filter': False, 'lambda_l1': 0.00020684237234498896, 'lambda_l2': 0.0001515387242576751, 'min_child_samples': 20}


In [12]:
train_predict = avg5_model.predict(X_train)
test_predict = avg5_model.predict(X_test)
MAE_train  =mean_absolute_error(y_train, train_predict)
MAE_test  =mean_absolute_error(y_test, test_predict)
R2_train = r2_score(y_train, train_predict)
R2_test = r2_score(y_test, test_predict)
result = {
    "model":"LightGBM",
    "dataset":"mv_avg_5",
    "MAE_train":mean_absolute_error(y_train, train_predict),
    "MAE_test":mean_absolute_error(y_test, test_predict),
    "R^2_train":r2_score(y_train, train_predict),
    "R^2_test":r2_score(y_test, test_predict)
}

pprint(result)
result_list.append(result)

{'MAE_test': 0.8927641330424779,
 'MAE_train': 0.6664479253800578,
 'R^2_test': 0.9132890115123279,
 'R^2_train': 0.9581832488647064,
 'dataset': 'mv_avg_5',
 'model': 'LightGBM'}


In [13]:
#2行のデータの移動平均を使用したデータでトライ
path_avg2 = '../data/processed/mv_avg_2.csv'
df = pd.read_csv(path_avg2)

X = df.drop(["tool_wear"], axis=1)
y = df["tool_wear"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_test, X_val, y_test, y_val = train_test_split(X_test, y_test, test_size=0.5, random_state=42)

trains = lgb.Dataset(X_train, y_train)
tests = lgb.Dataset(X_test, y_test)

model = lgb_tuner.LightGBMTuner(
    params, trains,valid_sets=[tests],
    num_boost_round=num_round,)

model.run()

best_params = model.best_params
print("Best params:", best_params)

avg2_model = model.get_best_booster()

[I 2025-11-05 11:36:17,526] A new study created in memory with name: no-name-cd9db89a-d71b-4b58-a501-f92ce7d866d7
feature_fraction, val_score: inf:   0%|                   | 0/7 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000220 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 1.581950:  14%|8     | 1/7 [00:00<00:01,  3.15it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000137 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 1.581950:  29%|#7    | 2/7 [00:00<00:01,  3.21it/s]

...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000098 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572


min_child_samples, val_score: 1.522581:  80%|#### | 4/5 [00:00<00:00,  4.21it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000112 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1275
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 5
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, be

min_child_samples, val_score: 1.522581: 100%|#####| 5/5 [00:01<00:00,  4.56it/s]

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Best params: {'objective': 'regression', 'metric': 'l1', 'num_leaves': 35, 'learning_rate': 0.1, 'fe

In [14]:
train_predict = avg2_model.predict(X_train)
test_predict = avg2_model.predict(X_test)
MAE_train  =mean_absolute_error(y_train, train_predict)
MAE_test  =mean_absolute_error(y_test, test_predict)
R2_train = r2_score(y_train, train_predict)
R2_test = r2_score(y_test, test_predict)
result = {
    "model":"LightGBM",
    "dataset":"mv_avg_2",
    "MAE_train":mean_absolute_error(y_train, train_predict),
    "MAE_test":mean_absolute_error(y_test, test_predict),
    "R^2_train":r2_score(y_train, train_predict),
    "R^2_test":r2_score(y_test, test_predict)
}

pprint(result)
result_list.append(result)

{'MAE_test': 1.5225813620857347,
 'MAE_train': 1.2049377556073315,
 'R^2_test': 0.7486970990308043,
 'R^2_train': 0.858253089819978,
 'dataset': 'mv_avg_2',
 'model': 'LightGBM'}


In [15]:
#MAEとR^2の推移を表示
result_df = pd.DataFrame(result_list)
lgb_results=result_df[result_df["model"]=="LightGBM"]

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
sns.barplot(lgb_results, x='dataset', y='MAE_test', errorbar=None)
plt.title('MAE')
plt.xticks(rotation=45)
plt.tight_layout()

plt.subplot(1,2,2)
sns.barplot(lgb_results, x='dataset', y='R^2_test', errorbar=None)
plt.xticks(rotation=45)
plt.title('R^2')
plt.tight_layout()

plt.savefig("../outputs/figures/modeling/raw_mvavg_result_histplot.png", format="png")
plt.close()

lgb_results[["dataset", "MAE_test", "R^2_test"]]

,dataset,MAE_test,R^2_test
1,raw_data,1.827849,0.633593
2,mv_avg_20,0.241146,0.990886
3,mv_avg_10,0.564681,0.961233
4,mv_avg_5,0.892764,0.913289
5,mv_avg_2,1.522581,0.748697


過去のデータ数を多く使用したデータほど予測精度が良い(MAEが小さくR^2が大きい)モデルが作成できた。

次に各センサのデータに"pl_vib_vec"を追加したデータセットでトライする。

In [16]:
path_pvec = '../data/processed/add_plane_vec.csv'
df = pd.read_csv(path_pvec)

X = df.drop(["tool_wear"], axis=1)
y = df["tool_wear"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

trains = lgb.Dataset(X_train, y_train)
tests = lgb.Dataset(X_test, y_test)

model = lgb_tuner.LightGBMTuner(
    params, trains,valid_sets=[tests],
    num_boost_round=num_round,)

model.run()

best_params = model.best_params
print("Best params:", best_params)

add_pvec_model = model.get_best_booster()

[I 2025-11-05 11:36:52,980] A new study created in memory with name: no-name-3c1125f2-95e6-4ca1-88c5-3c3592254fd5
feature_fraction, val_score: inf:   0%|                   | 0/7 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000236 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1530
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 6
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 1.966486:  14%|8     | 1/7 [00:00<00:01,  3.16it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000125 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1530
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 6
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 1.950997:  29%|#7    | 2/7 [00:00<00:01,  3.21it/s]

...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000100 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1530
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 6
[LightGBM] [Info] Start training from score 6.951572


min_child_samples, val_score: 1.838312:  60%|###  | 3/5 [00:00<00:00, 17.28it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000112 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1530
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 6
[LightGBM] [Info] Start training from score 6.951572


min_child_samples, val_score: 1.838312: 100%|#####| 5/5 [00:00<00:00, 17.26it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000074 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1530
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 6
[LightGBM] [Info] Start training from score 6.951572
Best params: {'objective': 'regression', 'metric': 'l1', 'num_leaves': 4, 'learning_rate': 0.1, 'feature_fraction': 0.8999999999999999, 'bagging_freq': 1, 'bagging_fraction': 0.8829139734757978, 'random_state': 0, 'feature_pre_filter': False, 'lambda_l1': 0.09163475251017875, 'lambda_l2': 7.608169063139604e-05, 'min_child_samples': 20}


In [17]:
train_predict = add_pvec_model.predict(X_train)
test_predict = add_pvec_model.predict(X_test)
MAE_train  =mean_absolute_error(y_train, train_predict)
MAE_test  =mean_absolute_error(y_test, test_predict)
R2_train = r2_score(y_train, train_predict)
R2_test = r2_score(y_test, test_predict)
result = {
    "model":"LightGBM",
    "dataset":"add_plane_vec",
    "MAE_train":mean_absolute_error(y_train, train_predict),
    "MAE_test":mean_absolute_error(y_test, test_predict),
    "R^2_train":r2_score(y_train, train_predict),
    "R^2_test":r2_score(y_test, test_predict)
}

pprint(result)
result_list.append(result)

{'MAE_test': 1.8383122536986667,
 'MAE_train': 1.651477577801363,
 'R^2_test': 0.6355814378681818,
 'R^2_train': 0.7376257353667539,
 'dataset': 'add_plane_vec',
 'model': 'LightGBM'}


In [18]:
regressor_model = lgb.LGBMRegressor()
regressor_model._Booster = add_pvec_model
regressor_model._n_features = X_test.shape[1]
regressor_model.fitted_ = True
results_2 = permutation_importance(regressor_model, X_test, y_test, n_repeats=10, random_state=42)

importances_add_pvec_df=pd.DataFrame(zip(X.columns, add_pvec_model.feature_importance(importance_type='gain'), results_2['importances'].mean(axis=1)), columns=['features', 'feature_importance', 'permutation_importance'])

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
sns.barplot(importances_add_pvec_df, x='features', y='feature_importance', errorbar=None)
plt.xticks(rotation=45)
plt.title('Feature_importance')
plt.tight_layout()

plt.subplot(1,2,2)
sns.barplot(importances_add_pvec_df, x='features', y='permutation_importance', errorbar=None)
plt.xticks(rotation=45)
plt.title('Permutation_importance')
plt.tight_layout()

plt.savefig("../outputs/figures/modeling/add_pvec_importances_histplot.png", format="png")
plt.close()

importances_add_pvec_df

,features,feature_importance,permutation_importance
0,pl_vib_vec,2674.762880,0.019655
1,vibration_x,1295.442610,0.004707
2,vibration_y,280.057139,0.000057
3,vibration_z,651.381120,0.005568
4,acoustic_rms,46633.835768,0.901342
5,spindle_load,6986.272425,0.077598


In [19]:
result_df = pd.DataFrame(result_list)
con_df = pd.concat([result_df[(result_df["model"]=="LightGBM")&(result_df["dataset"]=="raw_data")],
           result_df[result_df["dataset"]=="add_plane_vec"]])

In [20]:
#MAEとR^2の推移を表示
result_df = pd.DataFrame(result_list)
con_df = pd.concat([result_df[(result_df["model"]=="LightGBM")&(result_df["dataset"]=="raw_data")],
           result_df[result_df["dataset"]=="add_plane_vec"]])

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
sns.barplot(con_df, x='dataset', y='MAE_test', errorbar=None)
plt.title('MAE')
plt.tight_layout()

plt.subplot(1,2,2)
sns.barplot(con_df, x='dataset', y='R^2_test', errorbar=None)
plt.title('R^2')
plt.tight_layout()

plt.savefig("../outputs/figures/modeling/raw_add_pvec_result_histplot.png", format="png")
plt.close()

con_df[["dataset", "MAE_test", "R^2_test"]]

,dataset,MAE_test,R^2_test
1,raw_data,1.827849,0.633593
6,add_plane_vec,1.838312,0.635581


生データと"pl_vib_vec"を追加したデータセットでMAEもR^2もほぼ変わらなかった(MAE変化率 0.6%)。特徴量の重要度評価からも、追加した特徴量はあまりモデルの予測精度には寄与しなかったことが分かる。

複数回の加工のデータの移動平均を使用することでモデルの予測精度を高めることができたが、１回の加工単位ではセンサー値のブレが大きく、特徴量化が難しいと考える。

続いて、必要なセンサを確認するため、重要度評価が低いセンサのデータを削減してモデルを再度作成する。使用するデータセットは、生データと作成した中で最も予測精度の良いモデルを作成できた、20回の加工データの移動平均をとったデータセット。

In [21]:
#生データから重要度評価が最も低い特徴量を削除して再評価
path = '../data/raw/Milling_Tool_Dataset.csv'
df = pd.read_csv(path)

X = df[["vibration_x", "vibration_y", "acoustic_rms", "spindle_load"]]
y = df["tool_wear"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

trains = lgb.Dataset(X_train, y_train)
tests = lgb.Dataset(X_test, y_test)

model = lgb_tuner.LightGBMTuner(
    params, trains,valid_sets=[tests],
    num_boost_round=num_round,)

model.run()

best_params = model.best_params
print("Best params:", best_params)

z_del_model = model.get_best_booster()

[I 2025-11-05 11:39:22,040] A new study created in memory with name: no-name-f720e507-e0ca-429d-806e-b1f21913da1e
feature_fraction, val_score: inf:   0%|                   | 0/7 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000158 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1020
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 4
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 1.990633:  14%|8     | 1/7 [00:00<00:01,  3.15it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000109 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1020
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 4
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 1.990633:  29%|#7    | 2/7 [00:00<00:01,  3.23it/s]

...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000141 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1020
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 4
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000121 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1020
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 4
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000092 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1020
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 4
[LightGBM] [Info] Start training fro

min_child_samples, val_score: 1.843827:  80%|#### | 4/5 [00:00<00:00, 29.66it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000091 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1020
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 4
[LightGBM] [Info] Start training from score 6.951572


min_child_samples, val_score: 1.843827: 100%|#####| 5/5 [00:00<00:00, 29.38it/s]

Best params: {'objective': 'regression', 'metric': 'l1', 'num_leaves': 2, 'learning_rate': 0.1, 'feature_fraction': 1.0, 'bagging_freq': 1, 'bagging_fraction': 0.9361605961835495, 'random_state': 0, 'feature_pre_filter': False, 'lambda_l1': 0.0, 'lambda_l2': 0.0, 'min_child_samples': 5}


In [22]:
train_predict = z_del_model.predict(X_train)
test_predict = z_del_model.predict(X_test)
MAE_train  =mean_absolute_error(y_train, train_predict)
MAE_test  =mean_absolute_error(y_test, test_predict)
R2_train = r2_score(y_train, train_predict)
R2_test = r2_score(y_test, test_predict)
result = {
    "model":"LightGBM",
    "dataset":"raw_data_del_z",
    "MAE_train":mean_absolute_error(y_train, train_predict),
    "MAE_test":mean_absolute_error(y_test, test_predict),
    "R^2_train":r2_score(y_train, train_predict),
    "R^2_test":r2_score(y_test, test_predict)
}

pprint(result)
result_list.append(result)

{'MAE_test': 1.843827372713187,
 'MAE_train': 1.8043883683916624,
 'R^2_test': 0.6404436835600897,
 'R^2_train': 0.692699297429368,
 'dataset': 'raw_data_del_z',
 'model': 'LightGBM'}


In [23]:
#MAEが元データの5%下がるまで特徴量の削除を続行
X = df[["vibration_x", "acoustic_rms", "spindle_load"]]
y = df["tool_wear"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

trains = lgb.Dataset(X_train, y_train)
tests = lgb.Dataset(X_test, y_test)

model = lgb_tuner.LightGBMTuner(
    params, trains,valid_sets=[tests],
    num_boost_round=num_round,)

model.run()

best_params = model.best_params
print("Best params:", best_params)

yz_del_model = model.get_best_booster()

[I 2025-11-05 11:39:35,678] A new study created in memory with name: no-name-78c5327c-934a-4d30-ae75-238c4302b35e
feature_fraction, val_score: inf:   0%|                   | 0/7 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000193 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 3
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 1.963070:  14%|8     | 1/7 [00:00<00:01,  3.26it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000097 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 3
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 1.963070:  29%|#7    | 2/7 [00:00<00:01,  3.26it/s]

...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000115 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 3
[LightGBM] [Info] Start training from score 6.951572


min_child_samples, val_score: 1.831962:  80%|#### | 4/5 [00:00<00:00, 35.59it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000135 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 3
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000124 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 3
[LightGBM] [Info] Start training from score 6.951572


min_child_samples, val_score: 1.831962: 100%|#####| 5/5 [00:00<00:00, 35.10it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000105 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 765
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 3
[LightGBM] [Info] Start training from score 6.951572
Best params: {'objective': 'regression', 'metric': 'l1', 'num_leaves': 2, 'learning_rate': 0.1, 'feature_fraction': 1.0, 'bagging_freq': 4, 'bagging_fraction': 0.6387922431473659, 'random_state': 0, 'feature_pre_filter': False, 'lambda_l1': 8.591993068810066, 'lambda_l2': 0.58051901510462, 'min_child_samples': 5}


In [24]:
train_predict = yz_del_model.predict(X_train)
test_predict = yz_del_model.predict(X_test)
MAE_train  =mean_absolute_error(y_train, train_predict)
MAE_test  =mean_absolute_error(y_test, test_predict)
R2_train = r2_score(y_train, train_predict)
R2_test = r2_score(y_test, test_predict)
result = {
    "model":"LightGBM",
    "dataset":"raw_data_del_yz",
    "MAE_train":mean_absolute_error(y_train, train_predict),
    "MAE_test":mean_absolute_error(y_test, test_predict),
    "R^2_train":r2_score(y_train, train_predict),
    "R^2_test":r2_score(y_test, test_predict)
}

pprint(result)
result_list.append(result)

{'MAE_test': 1.8319615317501714,
 'MAE_train': 1.8296427285763375,
 'R^2_test': 0.6366809180312178,
 'R^2_train': 0.6823114097403178,
 'dataset': 'raw_data_del_yz',
 'model': 'LightGBM'}


In [25]:
X = df[["acoustic_rms", "spindle_load"]]
y = df["tool_wear"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

trains = lgb.Dataset(X_train, y_train)
tests = lgb.Dataset(X_test, y_test)

model = lgb_tuner.LightGBMTuner(
    params, trains,valid_sets=[tests],
    num_boost_round=num_round,)

model.run()

best_params = model.best_params
print("Best params:", best_params)

vib_del_model = model.get_best_booster()

[I 2025-11-05 11:39:59,970] A new study created in memory with name: no-name-b1bcd4cd-4b2d-45d8-a394-5ed4c4cdbd38
feature_fraction, val_score: inf:   0%|                   | 0/7 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000199 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 2
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 1.959869:  14%|8     | 1/7 [00:00<00:01,  3.13it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000100 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 2
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 1.959869:  29%|#7    | 2/7 [00:00<00:01,  3.16it/s]

...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000096 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 2
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000099 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 2
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000116 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 2
[LightGBM] [Info] Start training from s

min_child_samples, val_score: 1.863927:  80%|#### | 4/5 [00:00<00:00, 31.76it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000071 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 2
[LightGBM] [Info] Start training from score 6.951572


min_child_samples, val_score: 1.863927: 100%|#####| 5/5 [00:00<00:00, 31.71it/s]

Best params: {'objective': 'regression', 'metric': 'l1', 'num_leaves': 2, 'learning_rate': 0.1, 'feature_fraction': 1.0, 'bagging_freq': 2, 'bagging_fraction': 0.4992223866780922, 'random_state': 0, 'feature_pre_filter': False, 'lambda_l1': 0.6640085412557027, 'lambda_l2': 0.34416311636996877, 'min_child_samples': 50}


In [26]:
train_predict = vib_del_model.predict(X_train)
test_predict = vib_del_model.predict(X_test)
MAE_train  =mean_absolute_error(y_train, train_predict)
MAE_test  =mean_absolute_error(y_test, test_predict)
R2_train = r2_score(y_train, train_predict)
R2_test = r2_score(y_test, test_predict)
result = {
    "model":"LightGBM",
    "dataset":"raw_data_del_vib",
    "MAE_train":mean_absolute_error(y_train, train_predict),
    "MAE_test":mean_absolute_error(y_test, test_predict),
    "R^2_train":r2_score(y_train, train_predict),
    "R^2_test":r2_score(y_test, test_predict)
}

pprint(result)
result_list.append(result)

{'MAE_test': 1.8639270020606313,
 'MAE_train': 1.8846910823395477,
 'R^2_test': 0.6315439359723224,
 'R^2_train': 0.6696580954031066,
 'dataset': 'raw_data_del_vib',
 'model': 'LightGBM'}


In [27]:
from sklearn.linear_model import LinearRegression
X = df[['acoustic_rms']]
y = df['tool_wear']

model = LinearRegression()
model.fit(X, y)
y_predict = model.predict(X)

print(f"MAE: {mean_absolute_error(y, y_predict):.3f}")
print(f"R²: {r2_score(y, y_predict):.3f}")
print(f"回帰式: tool_wear = {model.coef_[0]:.4f} * acoustic_rms + {model.intercept_:.4f}")

plt.scatter(X, y, alpha=0.4, label='Actual')
plt.plot(X, y_predict, color='red', label='Regression line')
plt.xlabel('acoustic_rms')
plt.ylabel('tool_wear')
plt.legend()
plt.savefig("../outputs/figures/modeling/raw_acoustic_lineplot.png", format="png")
plt.close()

MAE: 2.105
R²: 0.587
回帰式: tool_wear = 40.3962 * acoustic_rms + -17.1735


生データから振動データ(vibration_x, vibration_y, vibration_z)を削除しても、予測精度はほとんど変わらなかった(MSE変化率 2.0%)。

さらに"spindle_load"を削除し、"acoustic_rms"のみを使用して単回帰分析を行った。この分析の評価は R^2=0.587, MAE=2.105であり、"acoustic_rms"のみでもある程度"tool_wear"を説明できている。

以上から、音響データ(acoustic_rms)が振動データやスピンドル負荷(spindle_load)データを内包している可能性が考えられる。

続いて、20回分の加工データの移動平均を使用したデータセットでも、振動データと"spindle_load"を除去してモデルを作成し、MAEの変化を確認する。

In [28]:
path_avg20 = '../data/processed/mv_avg_20.csv'
df = pd.read_csv(path_avg20)

X = df[["mv_avg_ar", "mv_avg_sl"]]
y = df["tool_wear"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

trains = lgb.Dataset(X_train, y_train)
tests = lgb.Dataset(X_test, y_test)

model = lgb_tuner.LightGBMTuner(
    params, trains,valid_sets=[tests],
    num_boost_round=num_round,)

model.run()

best_params = model.best_params
print("Best params:", best_params)

avg20_vib_del_model = model.get_best_booster()

[I 2025-11-05 11:43:17,177] A new study created in memory with name: no-name-9eeb35c8-9df6-4098-9002-7aa9da63928a
feature_fraction, val_score: inf:   0%|                   | 0/7 [00:00<?, ?it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000184 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 2
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 0.403093:  14%|8     | 1/7 [00:00<00:01,  3.13it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000121 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 2
[LightGBM] [Info] Start training from score 6.951572


feature_fraction, val_score: 0.403093:  29%|#7    | 2/7 [00:00<00:01,  3.19it/s]

...

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000060 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 2
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

min_child_samples, val_score: 0.397229:  80%|#### | 4/5 [00:00<00:00,  3.75it/s]

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000108 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 510
[LightGBM] [Info] Number of data points in the train set: 980, number of used features: 2
[LightGBM] [Info] Start training from score 6.951572
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, bes

min_child_samples, val_score: 0.397229: 100%|#####| 5/5 [00:01<00:00,  4.01it/s]

[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No f

In [29]:
train_predict = avg20_vib_del_model.predict(X_train)
test_predict = avg20_vib_del_model.predict(X_test)
MAE_train  =mean_absolute_error(y_train, train_predict)
MAE_test  =mean_absolute_error(y_test, test_predict)
R2_train = r2_score(y_train, train_predict)
R2_test = r2_score(y_test, test_predict)
result = {
    "model":"LightGBM",
    "dataset":"mv_avg20_del_vib",
    "MAE_train":mean_absolute_error(y_train, train_predict),
    "MAE_test":mean_absolute_error(y_test, test_predict),
    "R^2_train":r2_score(y_train, train_predict),
    "R^2_test":r2_score(y_test, test_predict)
}

pprint(result)
result_list.append(result)

{'MAE_test': 0.397228634576835,
 'MAE_train': 0.27194598660398084,
 'R^2_test': 0.9818588597083427,
 'R^2_train': 0.9923270377821269,
 'dataset': 'mv_avg20_del_vib',
 'model': 'LightGBM'}


In [30]:
X = df[['mv_avg_ar']]
y = df['tool_wear']

model = LinearRegression()
model.fit(X, y)
y_pred = model.predict(X)

print(f"MAE: {mean_absolute_error(y, y_pred):.3f}")
print(f"R²: {r2_score(y, y_pred):.3f}")
print(f"回帰式: tool_wear = {model.coef_[0]:.4f} * acoustic_rms + {model.intercept_:.4f}")

plt.scatter(X, y, alpha=0.4, label='Actual')
plt.plot(X, y_pred, color='red', label='Regression line')
plt.xlabel('acoustic_rms')
plt.ylabel('tool_wear')
plt.legend()
plt.savefig("../outputs/figures/modeling/mv_avg20_acoustic_lineplot.png", format="png")
plt.close()

MAE: 0.572
R²: 0.971
回帰式: tool_wear = 66.4680 * acoustic_rms + -32.6708


20回分の加工データの移動平均を取ったデータセットから振動データを削除し、モデルを作成した。評価値はR^2 =0.982, MAE=0.397。

次に"acoustic_rms"のみで単回帰分析を行うと、評価値は、R^2 =0.971, MAE=0.572。

MAEの変化率は、振動データを削除すると+64.7%、"spindle_load"も削除すると+137.3%。

このことから、高精度な予測をしようとすると全てのセンサが必要なことが確認できた。

ただし、振動データと"spindle_load"を両方削除したデータセットから作成されたモデルでもR^2=0.971と非常に高く、高い予測精度を求めなければ、音響センサ(acoustic_rms)のみでも十分予測精度が高いモデルが構築可能。

まとめ

工具摩耗値の予測を高精度にしようとするなら複数の加工データと全てのセンサのデータが必要である。 


加工データの数について、求められる加工精度にもよるため一概には言えないが、データ数5以上であればR^2が0.9以上の非常に予測精度が良いモデルが作成可能。

１回の加工データからの予測ではMAEは1.8〜1.9程度で、振動センサのデータを無くしてもほとんどモデルの予測精度には影響しない。

必要なセンサについて、音響センサだけでも20回分の加工データがあれば十分に精度の良いモデルは作成可能であり、高精度が必要でなければ音響センサのみでもよい。